# 模块十二：跨系统 DWA + DuckDB OLAP（Phase 2 / Background §6.12）

## 学习目标

1. 理解「跨主题 4 表 JOIN」的实际业务约束（SAP 销售订单不含 MINE_CODE 字段）
2. 掌握 DuckDB 4 表 LEFT JOIN + `CREATE TABLE AS SELECT` 物化模拟
3. 演练 4 个分析场景 SQL：产销对比 / 煤质定价 / 安全趋势 / 订单履约

## 步骤 1：跑 `build_dwa_sales_production.py` 生成 4 表 JOIN 宽表

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "../scripts/build_dwa_sales_production.py", "--sample", "0.01"],
    capture_output=True, text=True, cwd="..",
)
print(result.stdout[-1200:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## 步骤 2：4 个分析场景 SQL 示例（产销 / 煤质 / 安全 / 订单）

In [ ]:
import duckdb
from pathlib import Path

LAKEHOUSE = Path("..") / "data" / "lakehouse"
DWA = LAKEHOUSE / "dwa" / "dwa_sales_production"

conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE VIEW dwa AS SELECT * FROM delta_scan('{DWA}')")

print("=== 场景 1：产销对比 — 各矿井日均产量 vs 订单数 ===")
df1 = conn.execute("""
    SELECT mine_code, COUNT(DISTINCT VBELN) AS order_count,
           ROUND(AVG(daily_production), 2) AS avg_production,
           ROUND(AVG(NETWR), 2) AS avg_order_amount
    FROM dwa WHERE mine_code IS NOT NULL AND mine_code != ''
    GROUP BY mine_code ORDER BY avg_production DESC
    LIMIT 10
 """).df()
print(df1.to_string(index=False))

print("\n=== 场景 2：煤质定价 — 各矿井煤质 vs 订单均价 ===")
df2 = conn.execute("""
    SELECT mine_code,
           ROUND(AVG(ash_content), 2) AS avg_ash,
           ROUND(AVG(calorific), 2) AS avg_calorific,
           ROUND(AVG(NETWR), 2) AS avg_price
    FROM dwa WHERE mine_code IS NOT NULL AND mine_code != ''
          AND ash_content > 0
    GROUP BY mine_code ORDER BY avg_calorific DESC
    LIMIT 10
 """).df()
print(df2.to_string(index=False))

## 步骤 3：matplotlib 渲染 4 表 JOIN 关键指标图（产销对比）

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig, ax1 = plt.subplots(figsize=(10, 6))

mines = df1["mine_code"].tolist()
x = range(len(mines))

ax1.bar(x, df1["order_count"], color="#4C72B0", alpha=0.7, label="订单数")
ax1.set_xlabel("矿井")
ax1.set_ylabel("日均订单数", color="#4C72B0")
ax1.set_xticks(x)
ax1.set_xticklabels(mines)

ax2 = ax1.twinx()
ax2.plot(x, df1["avg_production"], color="#DD8452", marker="o", label="日均生产")
ax2.set_ylabel("日均生产值", color="#DD8452")

plt.title("6.12 跨系统 DWA — 产销对比（按矿井）")
fig.tight_layout()

out = Path("step_images")
out.mkdir(exist_ok=True)
png = out / "module12_4table_join.png"
plt.savefig(png, dpi=120)
print(f"saved: {png}")
from IPython.display import Image, display
display(Image(filename=str(png)))